# DSC ML Arena: exploration and two-target modeling

A self-contained Python notebook for inspecting the competition data, training five-fold CatBoost baselines, and calibrating the rapid gamma-GT measurement by cartridge batch.

Predict `Smoking` probabilities and `Gamma_GT` values. Use only the three provided competition CSVs. Scores shown below are local exploratory validation results.

Run cells in order. Training may take several minutes. Outputs go to `outputs/notebook/`, preserving the earlier script results. No cell uploads a submission.

Select the project `.venv` Python kernel. If dependencies are missing, run in a terminal:
```bash
.venv/bin/python -m pip install -r requirements.txt nbformat nbclient ipykernel
```
Place the CSVs in `data/` next to this notebook. For Kaggle, change `DATA_DIR` to the attached competition dataset directory.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import sys
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from catboost import CatBoostClassifier, CatBoostRegressor

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs/notebook')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
FOLDS = 5
ITERATIONS = 2000
print('Python:', sys.version.split()[0])
versions = {p: metadata.version(p) for p in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'catboost', 'matplotlib']}
print(versions)
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(versions, indent=2));

## Load and verify the files

Both targets are excluded from inputs. ID is used to align output rows, not as a predictor. Exact duplicate checks do not rule out near duplicates.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
targets = ['Smoking', 'Gamma_GT']
features = [c for c in test.columns if c != 'ID']
assert set(train.columns) == set(test.columns) | set(targets)
assert train.ID.is_unique and test.ID.is_unique and sample.ID.is_unique
assert len(test) == 6399 and sample.ID.equals(test.ID)
assert set(train.Smoking.unique()) == {0, 1}
assert train[targets].notna().all().all()
assert np.isfinite(train.Gamma_GT).all() and train.Gamma_GT.ge(0).all()
integrity = {'training_rows': len(train), 'test_rows': len(test), 'predictors': len(features),
             'id_overlap': len(set(train.ID) & set(test.ID)),
             'duplicate_training_features': int(train[features].duplicated().sum()),
             'duplicate_test_features': int(test[features].duplicated().sum()),
             'cross_split_feature_hash_matches': int(pd.util.hash_pandas_object(test[features], index=False).isin(pd.util.hash_pandas_object(train[features], index=False)).sum())}
display(pd.Series(integrity, name='Value'))
display(train.head())
display(train.dtypes.rename('dtype').to_frame())

## Targets and missingness

AUC rewards smoking ranking quality. Gamma-GT is right-skewed; fit `log1p` values to align with RMSLE. Retain extreme target values rather than automatically deleting them.

In [ ]:
display(train.Smoking.value_counts().rename('Rows').to_frame())
print(f'Smoking prevalence: {train.Smoking.mean():.2%}')
display(train.Gamma_GT.describe(percentiles=[.5, .9, .95, .99]).to_frame())
missing = pd.DataFrame({'train_pct': train[features].isna().mean()*100,
                        'test_pct': test[features].isna().mean()*100}).sort_values('train_pct', ascending=False)
missing.to_csv(OUTPUT_DIR / 'missingness.csv')
display(missing.head(12).round(2))
fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
axes[0].hist(train.Gamma_GT, bins=70)
axes[0].set(title='Gamma-GT', xlabel='U/L', ylabel='Rows')
axes[1].hist(np.log1p(train.Gamma_GT), bins=50)
axes[1].set(title='Log target', xlabel='log1p(Gamma-GT)')
missing.head(10).iloc[::-1].plot.barh(ax=axes[2])
axes[2].set(title='Missing predictors', xlabel='Missing (%)')
plt.show()

## Center collection patterns and train/test groups

Missingness can reflect how records were collected. Group overlap determines whether validation measures interpolation within known groups or generalization to new groups.

In [ ]:
center_missing = pd.DataFrame({c: train[c].isna().groupby(train.Screening_Center).mean()*100
                               for c in ['Exhaled_CO_ppm', 'Alcohol_Units_Week', 'POC_GGT']})
display(center_missing.round(2))
groups = []
for c in ['Workplace_ID', 'Urine_Plate', 'POC_Batch', 'Screening_Center', 'Visit_Quarter']:
    observed = test[c].dropna()
    groups.append({'feature': c, 'train_categories': train[c].nunique(), 'test_categories': test[c].nunique(),
                   'unseen_test_nonmissing_pct': (~observed.isin(train[c].dropna())).mean()*100})
display(pd.DataFrame(groups).round(3))
shifts = []
for c in train[features].select_dtypes('number'):
    a, b = train[c].dropna(), test[c].dropna()
    shifts.append({'feature': c, 'KS': ks_2samp(a, b).statistic,
                   'train_median': a.median(), 'test_median': b.median()})
display(pd.DataFrame(shifts).sort_values('KS', ascending=False).head(10))

## Descriptive feature signals and redundancy

Single-feature AUCs below use all observed training labels and differing subsets. They are exploratory diagnostics, not held-out scores. Correlations are associations, not causal conclusions.

In [ ]:
signals = []
for c in train[features].select_dtypes('number'):
    observed = train[c].notna()
    if train.loc[observed, c].nunique() > 1:
        auc = roc_auc_score(train.loc[observed, 'Smoking'], train.loc[observed, c])
        signals.append({'feature': c, 'observed_rows': observed.sum(), 'raw_auc': auc,
                        'direction_adjusted_auc': max(auc, 1-auc)})
display(pd.DataFrame(signals).sort_values('direction_adjusted_auc', ascending=False).head(12))
corr = train[features].select_dtypes('number').corrwith(train.Gamma_GT, method='spearman')
display(corr.sort_values(ascending=False).head(12).rename('Gamma Spearman').to_frame())
ratio = train.Urine_Cotinine_ng_mL / train.Urine_Creatinine
for label, values in [('Raw cotinine', train.Urine_Cotinine_ng_mL), ('Cotinine / creatinine', ratio), ('Exhaled CO', train.Exhaled_CO_ppm)]:
    observed = values.notna() & np.isfinite(values)
    print(label, 'AUC:', round(roc_auc_score(train.Smoking[observed], values[observed]), 4), 'n:', observed.sum())
lipid_residual = train.LDL - (train.Cholesterol_Total - train.HDL - train.Triglyceride/5)
print('LDL formula absolute residual:', lipid_residual.abs().quantile([.5, .95]).to_dict())
print('Eyesight 9.9 codes:', {c: {'train': int(train[c].eq(9.9).sum()), 'test': int(test[c].eq(9.9).sum())} for c in ['Eyesight_L', 'Eyesight_R']})

## Rapid-test batch effects

Inspect the log ratio of true gamma-GT to the rapid reading. Batch offsets are descriptive here. Later, every validation correction will be estimated using only that fold's training rows.

In [ ]:
log_offset = np.log(train.Gamma_GT) - np.log(train.POC_GGT.where(train.POC_GGT.gt(0)))
batch_stats = log_offset.groupby(train.POC_Batch).agg(['count', 'mean', 'std'])
print('Between-batch offset std:', batch_stats['mean'].std())
print('Median within-batch std:', batch_stats['std'].median())
print('Median training rows per batch:', batch_stats['count'].median())
batch_stats['mean'].hist(bins=40, figsize=(8, 3))
plt.xlabel('Mean log(true gamma / rapid-test gamma)')
plt.ylabel('Batches')
plt.title('Batch offsets')
plt.show()

## Competition metric and preprocessing

Numeric missing values are retained for CatBoost. Identifier-like features become categories. No global target-derived transformation is added to the classifier.

In [ ]:
def score(y_smoke, y_gamma, smoke, gamma):
    auc = roc_auc_score(y_smoke, smoke)
    rmsle = np.sqrt(np.mean((np.log1p(y_gamma) - np.log1p(np.clip(gamma, 1, 1000))) ** 2))
    return {"auc": float(auc), "rmsle": float(rmsle),
            "score": float(0.5 * (2 * auc - 1) + 0.5 * (1 - rmsle / 0.5041))}

In [ ]:
args = SimpleNamespace(data=DATA_DIR, output=OUTPUT_DIR / 'baseline', folds=FOLDS, iterations=ITERATIONS, seed=SEED)
x, xt = train[features].copy(), test[features].copy()
# Identifier-like fields are categorical, never continuous measurements.
categorical = [c for c in features if x[c].dtype == 'object' or c in
               ['Urine_Plate', 'POC_Batch', 'Workplace_ID', 'Screening_Center', 'Visit_Quarter']]
for c in features:
    if c in categorical:
        x[c] = x[c].fillna('__MISSING__').astype(str)
        xt[c] = xt[c].fillna('__MISSING__').astype(str)
    else:
        x[c] = pd.to_numeric(x[c], errors='raise').replace([np.inf, -np.inf], np.nan)
        xt[c] = pd.to_numeric(xt[c], errors='raise').replace([np.inf, -np.inf], np.nan)

## Train the five-fold baseline

Five stratified shuffled folds, depth 6, learning rate 0.04, up to 2,000 iterations, and early stopping after 150 rounds without improvement. This cell fits all ten models. Early stopping uses validation labels, so these scores remain exploratory.

In [ ]:
ys = train.Smoking.to_numpy()
yg = np.log1p(train.Gamma_GT.to_numpy())
oof_s, oof_g = np.zeros(len(x)), np.zeros(len(x))
pred_s, pred_g_log = np.zeros(len(xt)), np.zeros(len(xt))
args.output.mkdir(parents=True, exist_ok=True)
fold_metrics = []
cv = StratifiedKFold(args.folds, shuffle=True, random_state=args.seed)
for fold, (tr, va) in enumerate(cv.split(x, ys)):
    common = dict(iterations=args.iterations, depth=6, learning_rate=0.04,
                  random_seed=args.seed + fold, cat_features=categorical,
                  verbose=False, allow_writing_files=False, thread_count=4)
    classifier = CatBoostClassifier(**common, loss_function='Logloss', eval_metric='AUC')
    regressor = CatBoostRegressor(**common, loss_function='RMSE')
    classifier.fit(x.iloc[tr], ys[tr], eval_set=(x.iloc[va], ys[va]), early_stopping_rounds=150)
    regressor.fit(x.iloc[tr], yg[tr], eval_set=(x.iloc[va], yg[va]), early_stopping_rounds=150)
    oof_s[va] = classifier.predict_proba(x.iloc[va])[:, 1]
    oof_g[va] = np.clip(np.expm1(regressor.predict(x.iloc[va])), 1, 1000)
    pred_s += classifier.predict_proba(xt)[:, 1] / args.folds
    pred_g_log += regressor.predict(xt) / args.folds
    metrics = score(ys[va], train.Gamma_GT.to_numpy()[va], oof_s[va], oof_g[va])
    fold_metrics.append(metrics)
    print(f'Fold {fold + 1}: {metrics}', flush=True)
    classifier.save_model(str(args.output / f'smoking_fold{fold}.cbm'))
    regressor.save_model(str(args.output / f'gamma_fold{fold}.cbm'))

## Save baseline outputs

Average classification probabilities across folds; average gamma predictions in log space. Record input hashes and fold metrics for reproduction.

In [ ]:
predictions = pd.DataFrame({'ID': test.ID, 'Smoking': pred_s,
                            'Gamma_GT': np.clip(np.expm1(pred_g_log), 1, 1000)})
submission = sample[['ID']].merge(predictions, on='ID', how='left', validate='one_to_one')
assert submission.ID.tolist() == sample.ID.tolist()
assert np.isfinite(submission[targets].to_numpy()).all()
assert submission.Smoking.between(0, 1).all()
submission.to_csv(args.output / 'submission.csv', index=False)
pd.DataFrame({'ID': train.ID, 'Smoking': oof_s, 'Gamma_GT': oof_g}).to_csv(args.output / 'oof.csv', index=False)
report = {'overall': score(ys, train.Gamma_GT.to_numpy(), oof_s, oof_g),
          'folds': fold_metrics, 'seed': args.seed, 'iterations': args.iterations,
          'features': features, 'categorical': categorical,
          'data_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                          for p in args.data.glob('*.csv')}}
(args.output / 'metrics.json').write_text(json.dumps(report, indent=2))
print(report['overall'])

## Fold-safe batch calibration

For each held-out fold, estimate batch offsets exclusively from the other folds. Missing or nonpositive readings and unknown batches retain their baseline prediction. The final test correction uses all training labels.

In [ ]:
def batch_predictions(reference, frame):
    valid = reference.POC_GGT.gt(0) & reference.Gamma_GT.gt(0) & reference.POC_Batch.notna()
    ref = reference.loc[valid]
    offsets = (np.log(ref.Gamma_GT) - np.log(ref.POC_GGT)).groupby(ref.POC_Batch).mean()
    correction = frame.POC_Batch.map(offsets)
    usable = frame.POC_GGT.gt(0) & correction.notna()
    pred = np.full(len(frame), np.nan)
    pred[usable] = np.clip(np.exp(np.log(frame.loc[usable, 'POC_GGT']) + correction[usable]), 1, 1000)
    return pred, offsets

In [ ]:
baseline_oof = pd.read_csv(args.output / 'oof.csv')
baseline_submission = pd.read_csv(args.output / 'submission.csv')
corrected = baseline_oof.Gamma_GT.to_numpy().copy()
coverage = np.zeros(len(train), dtype=bool)
for fit, hold in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(train, train.Smoking):
    p, _ = batch_predictions(train.iloc[fit], train.iloc[hold])
    usable = np.isfinite(p)
    corrected[hold[usable]] = p[usable]
    coverage[hold[usable]] = True
calibrated_test, offsets = batch_predictions(train, test)
usable_test = np.isfinite(calibrated_test)
final_submission = baseline_submission.copy()
final_submission.loc[usable_test, 'Gamma_GT'] = calibrated_test[usable_test]
results = pd.DataFrame({
    'Baseline': score(train.Smoking, train.Gamma_GT, baseline_oof.Smoking, baseline_oof.Gamma_GT),
    'Batch corrected': score(train.Smoking, train.Gamma_GT, baseline_oof.Smoking, corrected)})
display(results)
print(f'Calibration coverage: OOF {coverage.mean():.2%}; test {usable_test.mean():.2%}')

## Second split check and submission validation

The second split tests the calibration mechanism alone on covered rows. It is not an independent evaluation of the complete pipeline. Final output checks enforce shape, ID order, finite values, and prediction ranges.

In [ ]:
reference, holdout = train_test_split(train, test_size=.2, stratify=train.Smoking, random_state=2026)
p, _ = batch_predictions(reference, holdout)
usable = np.isfinite(p)
covered_rmsle = np.sqrt(np.mean((np.log1p(holdout.Gamma_GT.to_numpy()[usable])-np.log1p(p[usable]))**2))
print('Second split: coverage', usable.mean(), 'covered RMSLE', covered_rmsle)
assert final_submission.columns.tolist() == ['ID', 'Smoking', 'Gamma_GT']
assert len(final_submission) == 6399 and final_submission.ID.equals(sample.ID)
assert np.isfinite(final_submission[targets].to_numpy()).all()
assert final_submission.Smoking.between(0, 1).all()
assert final_submission.Gamma_GT.between(1, 1000).all()
final_submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
offsets.rename('log_multiplier').to_csv(OUTPUT_DIR / 'batch_offsets.csv')
results.to_csv(OUTPUT_DIR / 'comparison.csv')
print('Validated submission:', (OUTPUT_DIR / 'submission.csv').resolve())
display(final_submission.head())

## Interpretation and remaining work

- Batch calibration materially improves the regression task; smoking predictions remain unchanged.
- Group overlap supports validation within known batches, but does not demonstrate generalization to new batches or employers.
- Feature diagnostics and method selection used these labels. Before extensive tuning, reserve an untouched holdout or use nested validation.
- Next experiments: urine-plate effects, robust calibration for small batches, errors on missing rapid tests, and nested cross-target stacking.
- Competition data includes synthetic features. These models are for this competition only.

The notebook writes a candidate CSV but does not submit it to Kaggle.

# Plan implementation: stronger validation and feature experiments

This section contains the complete Python experiment workflow. It preserves previous results under `outputs/plan_v1/`.

- Reserve a stratified 20% outer holdout with seed 20261009.
- Select features and calibration on three folds of the remaining 80%.
- Use inner splits for early stopping, then refit the selected number of trees on the full fitting partition.
- Compare raw features with urine logs, concentration ratios, and plate-centered ratios.
- Compare mean/median batch correction, shrinkage, and a log-space blend with regression.
- Freeze the configuration before evaluating the outer holdout.
- Fit a final five-fold ensemble and validate the CSV.

Earlier exploration used all training data, so the outer holdout is a prospective check rather than historically untouched data. Method selection is confined to development labels in this workflow. Later final CV includes development-selected rows and is diagnostic.


In [ ]:
import hashlib
import importlib.metadata
import json
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
OUT = Path('outputs/plan_v1')
SEED = 20261009
MAX_ITERATIONS = 1600
from IPython.display import display


## Transformations and model fitting

Plate references use fitting-partition feature values only. Evaluation labels never enter features or calibration. These helper functions are independent of the earlier notebook cells.

In [ ]:
def metrics(frame, smoking, gamma):
    auc = roc_auc_score(frame.Smoking, smoking)
    rmsle = np.sqrt(np.mean((np.log1p(frame.Gamma_GT) - np.log1p(np.clip(gamma, 1, 1000)))**2))
    return {'auc': float(auc), 'rmsle': float(rmsle), 'score': float(auc - rmsle / (2*.5041))}

In [ ]:
def features(reference, frame, variant):
    cols = [c for c in frame if c not in ['ID', 'Smoking', 'Gamma_GT']]
    x = frame[cols].copy()
    if variant == 'urine':
        for marker in ['Urine_Cotinine_ng_mL', 'Urine_EtG_ng_mL']:
            x['log_' + marker] = np.log1p(frame[marker])
            x['log_ratio_' + marker] = np.log(frame[marker] / frame.Urine_Creatinine)
            ref_ratio = np.log(reference[marker] / reference.Urine_Creatinine)
            plate_mean = ref_ratio.groupby(reference.Urine_Plate).mean()
            # Fit the unsupervised plate reference only on the fitting partition.
            x['plate_centered_' + marker] = x['log_ratio_' + marker] - frame.Urine_Plate.map(plate_mean).fillna(ref_ratio.mean())
    cats = [c for c in x if x[c].dtype == 'object' or c in ['Workplace_ID','Urine_Plate','POC_Batch']]
    for c in x:
        if c in cats:
            x[c] = x[c].fillna('__MISSING__').astype(str)
        else:
            x[c] = pd.to_numeric(x[c], errors='raise').replace([np.inf,-np.inf],np.nan)
    return x, cats

In [ ]:
def fit_models(reference, variant, seed):
    inner_fit, inner_stop = train_test_split(reference, test_size=.2, stratify=reference.Smoking, random_state=seed)
    assert set(inner_fit.ID).isdisjoint(inner_stop.ID)
    xi, cats = features(inner_fit, inner_fit, variant)
    xs, _ = features(inner_fit, inner_stop, variant)
    models, iterations = [], []
    for kind in ['smoking', 'gamma']:
        cls = CatBoostClassifier if kind == 'smoking' else CatBoostRegressor
        extra = {'loss_function':'Logloss','eval_metric':'AUC'} if kind == 'smoking' else {'loss_function':'RMSE'}
        params = dict(depth=6, learning_rate=.04, random_seed=seed, cat_features=cats,
                      verbose=False, allow_writing_files=False, thread_count=4, **extra)
        yi = inner_fit.Smoking if kind == 'smoking' else np.log1p(inner_fit.Gamma_GT)
        ys = inner_stop.Smoking if kind == 'smoking' else np.log1p(inner_stop.Gamma_GT)
        pilot = cls(iterations=MAX_ITERATIONS, **params)
        pilot.fit(xi, yi, eval_set=(xs, ys), early_stopping_rounds=120)
        n = max(1, pilot.get_best_iteration()+1)
        xf, _ = features(reference, reference, variant)
        yf = reference.Smoking if kind == 'smoking' else np.log1p(reference.Gamma_GT)
        model = cls(iterations=n, **params)
        model.fit(xf, yf)
        models.append(model)
        iterations.append(n)
    return models, iterations

In [ ]:
def predict_models(models, reference, frame, variant):
    x, _ = features(reference, frame, variant)
    return models[0].predict_proba(x)[:,1], np.clip(np.expm1(models[1].predict(x)),1,1000)

In [ ]:
def batch_correct(reference, frame, fallback, method='mean', shrink=0, weight=1):
    good = reference.POC_GGT.gt(0) & reference.Gamma_GT.gt(0) & reference.POC_Batch.notna()
    ref = reference.loc[good]
    r = np.log(ref.Gamma_GT)-np.log(ref.POC_GGT)
    grouped = r.groupby(ref.POC_Batch)
    offset = grouped.mean() if method == 'mean' else grouped.median()
    count = grouped.count()
    offset = (count*offset+shrink*r.mean())/(count+shrink)
    mapped = frame.POC_Batch.map(offset)
    usable = frame.POC_GGT.gt(0) & mapped.notna()
    p = np.asarray(fallback).copy()
    raw = np.clip(np.exp(np.log(frame.loc[usable,'POC_GGT'])+mapped[usable]),1,1000)
    p[usable] = np.expm1(weight*np.log1p(raw)+(1-weight)*np.log1p(p[usable]))
    return np.clip(p,1,1000), usable.to_numpy()

## Complete experiment runner

The runner saves split IDs, development comparisons, the frozen configuration, holdout predictions, error slices, models, and a validated submission. The cell below defines it; execution is controlled separately.

In [ ]:
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv('data/train.csv'); test = pd.read_csv('data/test.csv'); sample = pd.read_csv('data/sample_submission.csv')
    dev, holdout = train_test_split(train, test_size=.2, stratify=train.Smoking, random_state=SEED)
    dev=dev.reset_index(drop=True);holdout=holdout.reset_index(drop=True)
    assert set(dev.ID).isdisjoint(holdout.ID)
    pd.concat([dev[['ID']].assign(partition='development'),holdout[['ID']].assign(partition='holdout')]).to_csv(OUT/'split_ids.csv',index=False)
    # All candidate selection happens on development folds before holdout evaluation.
    variants=['raw','urine']
    corrections=[dict(method='mean',shrink=0,weight=1),dict(method='median',shrink=0,weight=1),
                 dict(method='mean',shrink=.5,weight=1),dict(method='mean',shrink=2,weight=1),
                 dict(method='mean',shrink=0,weight=.9),dict(method='median',shrink=0,weight=.9)]
    records=[]
    predictions={}
    for variant in variants:
        smoke=np.zeros(len(dev));fallback=np.zeros(len(dev));corrected=[np.zeros(len(dev)) for _ in corrections]
        for fold,(fit,valid) in enumerate(StratifiedKFold(3,shuffle=True,random_state=SEED).split(dev,dev.Smoking)):
            ref=dev.iloc[fit]; frame=dev.iloc[valid]
            models,iters=fit_models(ref,variant,SEED+fold)
            s,g=predict_models(models,ref,frame,variant);smoke[valid]=s;fallback[valid]=g
            for i,c in enumerate(corrections): corrected[i][valid],_=batch_correct(ref,frame,g,**c)
            print('Development',variant,'fold',fold+1,'iterations',iters,metrics(frame,s,corrected[0][valid]),flush=True)
        records.append({'variant':variant,'correction':'none',**metrics(dev,smoke,fallback)})
        for i,c in enumerate(corrections):
            key=f'{variant}_{i}';record={'key':key,'variant':variant,**c,**metrics(dev,smoke,corrected[i])}
            records.append(record);predictions[key]=(smoke.copy(),corrected[i].copy())
    candidates=pd.DataFrame(records);candidates.to_csv(OUT/'development_comparison.csv',index=False)
    ranked=[r for r in records if 'key' in r]
    selected=max(ranked,key=lambda r:r['score'])
    variant=selected['variant']; correction={k:selected[k] for k in ['method','shrink','weight']}
    config={'variant':variant,'correction':correction,'seed':SEED,'max_iterations':MAX_ITERATIONS,
            'selection_metric':selected,'holdout_used_for_selection':False,
            'versions':{p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','catboost']},
            'data_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path('data').glob('*.csv')}}
    (OUT/'frozen_config.json').write_text(json.dumps(config,indent=2))
    print('FROZEN CONFIG',selected,flush=True)
    # Evaluate selected method once; keep configuration unchanged afterward.
    models,iters=fit_models(dev,variant,SEED+100)
    hs,hg=predict_models(models,dev,holdout,variant)
    hc,covered=batch_correct(dev,holdout,hg,**correction)
    hold_metrics={'regressor_only':metrics(holdout,hs,hg),'selected':metrics(holdout,hs,hc),
                  'coverage':float(covered.mean()),'iterations':iters,
                  'note':'Prospective holdout: full data was explored earlier; not historically untouched.'}
    if variant=='raw': os,og=hs,hg
    else:
        original,_=fit_models(dev,'raw',SEED+100)
        os,og=predict_models(original,dev,holdout,'raw')
    original_g,_=batch_correct(dev,holdout,og)
    hold_metrics['original_calibrated']=metrics(holdout,os,original_g)
    print('HOLDOUT',hold_metrics,flush=True)
    (OUT/'holdout_metrics.json').write_text(json.dumps(hold_metrics,indent=2))
    audit=holdout[['ID','Smoking','Gamma_GT','Screening_Center','POC_Batch']].copy()
    audit['smoking_prediction']=hs;audit['gamma_prediction']=hc;audit['calibrated']=covered
    audit['squared_log_error']=(np.log1p(audit.Gamma_GT)-np.log1p(hc))**2
    audit['gamma_range']=pd.cut(audit.Gamma_GT,[0,15,25,50,100,np.inf])
    counts=dev.POC_Batch.value_counts();audit['batch_training_n']=holdout.POC_Batch.map(counts).fillna(0)
    audit.to_csv(OUT/'holdout_predictions.csv',index=False)
    slices=[]
    for field in ['calibrated','Screening_Center','gamma_range']:
        for group,frame in audit.groupby(field,observed=True):
            m={'slice':field,'group':str(group),'rows':len(frame),
               'rmsle':float(np.sqrt(frame.squared_log_error.mean()))}
            m['auc']=float(roc_auc_score(frame.Smoking,frame.smoking_prediction)) if frame.Smoking.nunique()==2 else None
            slices.append(m)
    pd.DataFrame(slices).to_csv(OUT/'error_slices.csv',index=False)
    # Final five-fold ensemble using the frozen development-selected configuration.
    ps=np.zeros(len(test));pg_log=np.zeros(len(test));final_oof_s=np.zeros(len(train));final_oof_g=np.zeros(len(train))
    for fold,(fit,valid) in enumerate(StratifiedKFold(5,shuffle=True,random_state=SEED).split(train,train.Smoking)):
        ref=train.iloc[fit];frame=train.iloc[valid]
        models,iters=fit_models(ref,variant,SEED+200+fold)
        vs,vg=predict_models(models,ref,frame,variant);vg,_=batch_correct(ref,frame,vg,**correction)
        final_oof_s[valid]=vs;final_oof_g[valid]=vg
        ts,tg=predict_models(models,ref,test,variant);tg,_=batch_correct(ref,test,tg,**correction)
        ps+=ts/5;pg_log+=np.log1p(tg)/5
        for name,model in zip(['smoking','gamma'],models):model.save_model(str(OUT/f'{name}_fold{fold}.cbm'))
        print('Final fold',fold+1,metrics(frame,vs,vg),flush=True)
    submission=pd.DataFrame({'ID':test.ID,'Smoking':ps,'Gamma_GT':np.clip(np.expm1(pg_log),1,1000)})
    assert submission.ID.equals(sample.ID) and len(submission)==6399
    assert submission.columns.tolist()==['ID','Smoking','Gamma_GT']
    assert np.isfinite(submission[['Smoking','Gamma_GT']].to_numpy()).all()
    assert submission.Smoking.between(0,1).all() and submission.Gamma_GT.between(1,1000).all()
    submission.to_csv(OUT/'submission.csv',index=False)
    pd.DataFrame({'ID':train.ID,'Smoking':final_oof_s,'Gamma_GT':final_oof_g}).to_csv(OUT/'final_oof.csv',index=False)
    (OUT/'final_cv.json').write_text(json.dumps(metrics(train,final_oof_s,final_oof_g),indent=2))
    print('DONE',metrics(train,final_oof_s,final_oof_g),flush=True)

## Run or inspect saved results

Set `RUN_EXPERIMENTS = True` to retrain the entire workflow. The default reads the completed results to avoid repeating several minutes of training. Paths are relative to this notebook's directory.

In [ ]:
RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    main()
if not (OUT / 'holdout_metrics.json').exists():
    raise FileNotFoundError('Run the experiment workflow first by setting RUN_EXPERIMENTS = True.')
display(pd.read_csv(OUT / 'development_comparison.csv').sort_values('score', ascending=False))
print('Selected configuration:')
display(json.loads((OUT / 'frozen_config.json').read_text()))
print('Outer holdout results:')
display(json.loads((OUT / 'holdout_metrics.json').read_text()))
print('Error slices:')
display(pd.read_csv(OUT / 'error_slices.csv'))
if (OUT / 'final_cv.json').exists():
    print('Final diagnostic CV:')
    display(json.loads((OUT / 'final_cv.json').read_text()))


## Group-held-out robustness checks

These are diagnostics, not selection criteria. Holding out complete batches removes batch correction coverage; holding out workplaces tests transfer to unseen employers. They do not directly match the competition test's mostly shared groups.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
def run_group_diagnostics():
    cfg=json.loads((OUT/'frozen_config.json').read_text())
    data=pd.read_csv('data/train.csv')
    reports=[]
    for field in ['Workplace_ID','POC_Batch']:
        groups=data[field].fillna(-1)
        fit,hold=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(data,groups=groups))
        ref,frame=data.iloc[fit],data.iloc[hold]
        assert set(groups.iloc[fit]).isdisjoint(set(groups.iloc[hold]))
        models,iterations=fit_models(ref,cfg['variant'],SEED+500)
        s,g=predict_models(models,ref,frame,cfg['variant'])
        corrected,coverage=batch_correct(ref,frame,g,**cfg['correction'])
        result={'held_out_group':field,'fit_rows':len(ref),'holdout_rows':len(frame),
                'calibration_coverage':float(coverage.mean()),'iterations':iterations,
                **metrics(frame,s,corrected)}
        reports.append(result)
        print(result,flush=True)
    (OUT/'group_robustness.json').write_text(json.dumps(reports,indent=2))

In [ ]:
RUN_GROUP_DIAGNOSTICS = False
if RUN_GROUP_DIAGNOSTICS:
    run_group_diagnostics()
if (OUT / 'group_robustness.json').exists():
    display(pd.DataFrame(json.loads((OUT / 'group_robustness.json').read_text())))


## Candidate submission

This only validates and displays the file. It does not upload it to Kaggle. Keep method selection based on development experiments rather than repeatedly tuning to the public leaderboard.

In [ ]:
submission_path = OUT / 'submission.csv'
submission = pd.read_csv(submission_path)
sample = pd.read_csv('data/sample_submission.csv')
assert submission.columns.tolist() == ['ID', 'Smoking', 'Gamma_GT']
assert len(submission) == 6399 and submission.ID.equals(sample.ID)
assert np.isfinite(submission[['Smoking', 'Gamma_GT']].to_numpy()).all()
assert submission.Smoking.between(0, 1).all()
assert submission.Gamma_GT.between(1, 1000).all()
print('Validated candidate:', submission_path.resolve())
display(submission.head())
